<a href="https://colab.research.google.com/github/michael-alu/alu-ml-y2-t3-formative-3-group-1/blob/bayesian-probability/Group_1_Part_2_Bayesian_Probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import pandas as pd
import os

In [14]:
df = pd.read_csv("IMDB Dataset.csv")
print(df.shape)
print(df.head())
print(df['sentiment'].value_counts())

(50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [15]:
df['review'] = df['review'].str.lower()

positive_reviews = df[df['sentiment'] == 'positive']
negative_reviews = df[df['sentiment'] == 'negative']

total_reviews = len(df)
total_positive = len(positive_reviews)
total_negative = len(negative_reviews)

print(f"Total reviews: {total_reviews}")
print(f"Positive reviews: {total_positive}")
print(f"Negative reviews: {total_negative}")

Total reviews: 50000
Positive reviews: 25000
Negative reviews: 25000


In [16]:
positive_keywords = ["excellent","brilliant"]
negative_keywords = ["boring","terrible"]
all_keywords = positive_keywords + negative_keywords

### Keyword choice and what we measure

For this part, we decided to compute **P(Positive | keyword)**.

- **Positive keywords**: `"excellent"`, `"brilliant"`  
  These usually show up in positive reviews, so we expect them to **increase** the chance that a review is positive.
- **Negative keywords**: `"boring"`, `"terrible"`  
  These normally appear in negative reviews, so we expect them to **decrease** the chance that a review is positive.

For every keyword we report:

- **Prior**: `P(Positive)`  
- **Likelihood**: `P(keyword | Positive)`  
- **Marginal**: `P(keyword)` over the whole dataset  
- **Posterior**: `P(Positive | keyword)` using Bayes' Theorem.

In [17]:
def calculate_prior(total_positive: int, total_reviews: int) -> float:
    # P(Positive)
    return total_positive / total_reviews


def calculate_likelihood(keyword: str, subset_df: pd.DataFrame) -> float:
    # P(keyword | Positive) = reviews with keyword in positive subset / total positive reviews
    contains_keyword = subset_df["review"].str.contains(keyword, regex=False).sum()
    return contains_keyword / len(subset_df)


def calculate_marginal(keyword: str, df: pd.DataFrame) -> float:
    # P(keyword) = reviews with keyword in whole dataset / total reviews
    contains_keyword = df["review"].str.contains(keyword, regex=False).sum()
    return contains_keyword / len(df)


def calculate_posterior(prior: float, likelihood: float, marginal: float) -> float:
    # P(Positive | keyword) using Bayes: (P(kw|Pos) * P(Pos)) / P(kw)
    if marginal == 0:
        return 0.0
    return (likelihood * prior) / marginal


def analyze_keyword(
    keyword: str,
    df: pd.DataFrame,
    positive_reviews: pd.DataFrame,
    total_positive: int,
    total_reviews: int,
) -> dict:
    # Run full Bayesian analysis for one keyword
    prior = calculate_prior(total_positive, total_reviews)
    likelihood = calculate_likelihood(keyword, positive_reviews)
    marginal = calculate_marginal(keyword, df)
    posterior = calculate_posterior(prior, likelihood, marginal)

    return {
        "Keyword": keyword,
        "Prior P(Positive)": round(prior, 4),
        "Likelihood P(kw|Pos)": round(likelihood, 4),
        "Marginal P(kw)": round(marginal, 4),
        "Posterior P(Pos|kw)": round(posterior, 4),
    }

In [18]:
Keywords = positive_keywords + negative_keywords

results = []

for kw in all_keywords:
    result = analyze_keyword(kw, df, positive_reviews, total_positive, total_reviews)
    # mark whether we originally considered this a positive or negative signal
    result["Keyword Type"] = "positive" if kw in positive_keywords else "negative"
    results.append(result)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


  Keyword  Prior P(Positive)  Likelihood P(kw|Pos)  Marginal P(kw)  Posterior P(Pos|kw) Keyword Type
excellent                0.5                0.1174          0.0725               0.8099     positive
brilliant                0.5                0.0755          0.0489               0.7711     positive
   boring                0.5                0.0247          0.0623               0.1983     negative
 terrible                0.5                0.0154          0.0541               0.1419     negative


In [19]:
from IPython.display import display
display(results_df.style.set_caption("Bayesian Sentiment Analysis for IMDb Reviews")
                        .highlight_max(subset=["Posterior P(Pos|kw)"], color='lightgreen')
                        .highlight_min(subset=["Posterior P(Pos|kw)"], color='lightcoral'))


,Keyword,Prior P(Positive),Likelihood P(kw|Pos),Marginal P(kw),Posterior P(Pos|kw),Keyword Type
0,excellent,0.500000,0.117400,0.072500,0.809900,positive
1,brilliant,0.500000,0.075500,0.048900,0.771100,positive
2,boring,0.500000,0.024700,0.062300,0.198300,negative
3,terrible,0.500000,0.015400,0.054100,0.141900,negative
